In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss
)



In [2]:
# Load test data with PD
test_model = pd.read_csv(
    "../data/processed/champion_test_pd.csv"
)

# Load champion model
model = joblib.load(
    "../outputs/models/champion_logistic_model.pkl"
)

print("Test data shape:", test_model.shape)
print("Model loaded successfully.")

Test data shape: (30000, 11)
Model loaded successfully.


In [4]:
print(test_model.columns.tolist())

['Credit_History_Years_WOE', 'Credit_Utilization_WOE', 'Debt_to_Income_WOE', 'Employment_Length_Years_WOE', 'Monthly_Income_WOE', 'Number_of_Delinquencies_30_WOE', 'Number_of_Delinquencies_60_WOE', 'Number_of_Delinquencies_90_WOE', 'Bad', 'Risk_Band', 'Score']


In [5]:
woe_features = [
    "Credit_History_Years_WOE",
    "Credit_Utilization_WOE",
    "Debt_to_Income_WOE",
    "Employment_Length_Years_WOE",
    "Monthly_Income_WOE",
    "Number_of_Delinquencies_30_WOE",
    "Number_of_Delinquencies_60_WOE",
    "Number_of_Delinquencies_90_WOE"
]

X_test = test_model[woe_features]

test_pd = model.predict_proba(X_test)[:, 1]

test_model["PD"] = test_pd

print("PD created successfully.")
print(test_model["PD"].describe())

PD created successfully.
count    30000.000000
mean         0.068364
std          0.026381
min          0.028968
25%          0.051219
50%          0.063364
75%          0.078510
max          0.500604
Name: PD, dtype: float64


In [6]:
y_test = test_model["Bad"]

auc = roc_auc_score(y_test, test_pd)
gini = 2 * auc - 1

fpr, tpr, thresholds = roc_curve(y_test, test_pd)
ks = np.max(tpr - fpr)

print("Test AUC :", round(auc, 4))
print("Gini     :", round(gini, 4))
print("KS       :", round(ks, 4))

Test AUC : 0.5969
Gini     : 0.1938
KS       : 0.1455


In [7]:
# Calibration validation

brier = brier_score_loss(y_test, test_pd)

print("Brier Score :", round(brier, 4))

Brier Score : 0.063


In [8]:
# PD decile calibration

test_model["PD_Decile"] = pd.qcut(
    test_model["PD"],
    q=10,
    duplicates="drop"
)

calibration = (
    test_model.groupby("PD_Decile", observed=False)
    .agg(
        Average_PD=("PD", "mean"),
        Actual_Bad_Rate=("Bad", "mean"),
        Count=("Bad", "count")
    )
    .reset_index()
)

calibration["Difference"] = (
    calibration["Actual_Bad_Rate"] - calibration["Average_PD"]
)

calibration

,PD_Decile,Average_PD,Actual_Bad_Rate,Count,Difference
0,"(0.028, 0.043]",0.038660,0.044000,3000,0.005340
1,"(0.043, 0.0486]",0.045942,0.043000,3000,-0.002942
2,"(0.0486, 0.0538]",0.051195,0.050333,3000,-0.000862
3,"(0.0538, 0.0586]",0.056212,0.054000,3000,-0.002212
4,"(0.0586, 0.0634]",0.060925,0.061667,3000,0.000742
5,"(0.0634, 0.0686]",0.065907,0.070000,3000,0.004093
6,"(0.0686, 0.0747]",0.071552,0.071000,3000,-0.000552
7,"(0.0747, 0.0832]",0.078644,0.075975,3001,-0.002669
8,"(0.0832, 0.0984]",0.090084,0.095032,2999,0.004947
9,"(0.0984, 0.501]",0.124520,0.117333,3000,-0.007186


In [9]:
risk_band_validation = (
    test_model.groupby("Risk_Band")
    .agg(
        Customers=("Bad", "count"),
        Average_PD=("PD", "mean"),
        Actual_Bad_Rate=("Bad", "mean")
    )
    .reset_index()
)

risk_band_validation

,Risk_Band,Customers,Average_PD,Actual_Bad_Rate
0,High Risk,9605,0.083898,0.081624
1,Low Risk,3279,0.039056,0.043611
2,Medium Risk,15372,0.056827,0.056987
3,Very High Risk,1744,0.139601,0.139908


In [11]:
threshold = 0.08

y_pred = (test_pd >= threshold).astype(int)

cm = confusion_matrix(y_test, y_pred)

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Threshold :", threshold)
print("Confusion Matrix:\n", cm)
print("Precision :", round(precision, 4))
print("Recall    :", round(recall, 4))
print("F1 Score  :", round(f1, 4))

Threshold : 0.08
Confusion Matrix:
 [[21633  6320]
 [ 1334   713]]
Precision : 0.1014
Recall    : 0.3483
F1 Score  : 0.157


In [12]:
validation_summary = pd.DataFrame({
    "Metric": [
        "AUC",
        "Gini",
        "KS",
        "Brier Score",
        "PSI"
    ],
    "Value": [
        auc,
        gini,
        ks,
        brier,
        0.0006
    ]
})

validation_summary["Value"] = validation_summary["Value"].round(4)

validation_summary

,Metric,Value
0,AUC,0.5969
1,Gini,0.1938
2,KS,0.1455
3,Brier Score,0.0630
4,PSI,0.0006


In [13]:
validation_summary.to_csv(
    "../outputs/metrics/final_validation_metrics.csv",
    index=False
)

print("Final validation metrics saved successfully.")

Final validation metrics saved successfully.
